# Notebook-legal.com.br

[Dataset](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

In [ ]:
%pip install pandas duckdb pyarrow

In [ ]:
import pandas as pd
import duckdb
from time import perf_counter
import datetime

In [ ]:
arquivo = "yellow_tripdata_2026-01.parquet"

tempo_pandas = perf_counter()
df = pd.read_parquet(arquivo)
print(f"Pandas: {perf_counter() - tempo_pandas:.5f}s")

con = duckdb.connect()

tempo_duckdb = perf_counter()
con.execute(f"""
    CREATE TABLE corridas AS
    SELECT * FROM read_parquet ('{ arquivo }')
""")
print(f"DuckDB: {perf_counter() - tempo_duckdb:.5f}s")

razao = tempo_pandas / tempo_duckdb

print(f"\n\n{'DuckDB' if razao > 1 else 'Pandas'} foi {max(razao, 1/razao):.5f}x mais rápido")

In [ ]:
print(f"Número de Linhas no df: {len(df)}")

con.sql("SELECT COUNT(*) FROM corridas").show()

print(f"Colunas: {df.columns}")

con.sql("""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_name = 'corridas'
""").show()

In [ ]:
df = df[(df['tpep_pickup_datetime'] >= '2026-01-01') 
      & (df['tpep_dropoff_datetime'] < '2026-02-01')]

con.sql("""
    DELETE FROM corridas
    WHERE tpep_pickup_datetime < '2026-01-01'
    OR tpep_dropoff_datetime >= '2026-02-01';
""")

In [ ]:
print(f"Número de Linhas no df: {len(df)}")
con.sql("SELECT COUNT(*) FROM corridas").show()

# 1) Corridas por horário (sem IA)
Para cada hora de início da corrida (0 a 23, a partir de tpep_pickup_datetime), calcule o número de corridas, a distância média (trip_distance) e o valor médio pago (total_amount). Ordene pelo número de corridas, do maior para o menor.

In [ ]:
tempo_duckdb = perf_counter()
results = con.sql("""
        SELECT date_part('hour', tpep_pickup_datetime) AS hour_pickup, COUNT(*) AS n_corridas, MEAN(trip_distance), MEAN(total_amount)
        FROM corridas
        GROUP BY hour_pickup
        ORDER BY n_corridas DESC
        """).fetchall()
tempo_duckdb = perf_counter() - tempo_duckdb

print(f'DUCKDB -> {tempo_duckdb:.5f}s')
print('(hour, #, mean_distance, mean_price)')
for result in results:
        print(result)

df_ = df.copy()
tempo_pandas = perf_counter()
df_['hour'] = df_['tpep_pickup_datetime'].dt.hour
pandas_results = (
        df_.groupby("hour", as_index=False)
           .agg(
                   n_corridas=('hour', 'size'),
                   mean_distance=('trip_distance', 'mean'),
                   mean_price=('total_amount', 'mean')
           )
           .sort_values('n_corridas', ascending=False)
)
tempo_pandas = perf_counter() - tempo_pandas
print(f'\n\nPANDAS -> {tempo_pandas:.5f}s')
print(pandas_results)

razao = tempo_pandas / tempo_duckdb

print(f"\n\n{'DuckDB' if razao > 1 else 'Pandas'} foi {max(razao, 1/razao):.5f}x mais rápido")

# 4) Consulta livre (com IA)
Utilize IA para propor uma pergunta analítica diferente das anteriores e auxiliar na implementação em Pandas e SQL. Informe a pergunta escolhida e registre os principais prompts utilizados.

In [ ]:
def consulta4_pandas(df):
    d = df[
        (df["tpep_pickup_datetime"] >= "2026-01-01")
        & (df["tpep_pickup_datetime"] < "2026-02-01")
        & (df["payment_type"] == 1)
        & (df["fare_amount"] > 0)
        & (df["trip_distance"] > 0)
    ].copy()

    d["faixa_distancia"] = pd.cut(
        d["trip_distance"],
        bins=[0, 2, 5, float("inf")],
        labels=["curta", "media", "longa"],
        right=False,
    )
    d["pct_gorjeta"] = d["tip_amount"] / d["fare_amount"] * 100

    res = (
        d.groupby("faixa_distancia", observed=True)
        .agg(
            n_corridas=("tip_amount", "size"),
            gorjeta_media=("tip_amount", "mean"),
            pct_gorjeta_medio=("pct_gorjeta", "mean"),
        )
        .round(2)
        .reset_index()
        .sort_values("faixa_distancia")
        .reset_index(drop=True)
    )
    res["faixa_distancia"] = res["faixa_distancia"].astype(str)
    return res

tempo_pandas = perf_counter()
res_pd = consulta4_pandas(df)
tempo_pandas = perf_counter() - tempo_pandas
print(res_pd)

SQL_Q4 = """
SELECT
    CASE
        WHEN trip_distance < 2 THEN 'curta'
        WHEN trip_distance < 5 THEN 'media'
        ELSE 'longa'
    END AS faixa_distancia,
    COUNT(*) AS n_corridas,
    ROUND(AVG(tip_amount), 2) AS gorjeta_media,
    ROUND(AVG(tip_amount / fare_amount * 100), 2) AS pct_gorjeta_medio
FROM corridas
WHERE tpep_pickup_datetime >= '2026-01-01'
  AND tpep_pickup_datetime <  '2026-02-01'
  AND payment_type = 1
  AND fare_amount > 0
  AND trip_distance > 0
GROUP BY faixa_distancia
ORDER BY CASE faixa_distancia
             WHEN 'curta' THEN 1
             WHEN 'media' THEN 2
             ELSE 3
         END
"""

tempo_duckdb = perf_counter()
res_sql = con.execute(SQL_Q4).df()
tempo_duckdb = perf_counter() - tempo_duckdb
print(res_sql)

razao = tempo_pandas / tempo_duckdb
print(f"Pandas: {tempo_pandas}s\nDuckDB: {tempo_duckdb}s")
print(f"\n\n{'DuckDB' if razao > 1 else 'Pandas'} foi {max(razao, 1/razao):.5f}x mais rápido")

# 5) Corridas simultâneas (sem IA).
Considere as corridas iniciadas no dia 15/01/2026 a partir das 8h. Conte quantos pares de corridas tiveram seus períodos de viagem sobrepostos, considerando os horários de início (tpep_pickup_datetime) e de término (tpep_dropoff_datetime). Considere apenas pares de corridas distintas e conte cada par uma única vez. Execute a consulta para três janelas de tempo, das 8h às 9h, das 8h às 10h e das 8h às 12h (sem incluir o horário final). Informe a quantidade de pares e o tempo de execução de cada janela. Se
alguma execução exceder a memória disponível, registre isso e não aumente a janela.

In [ ]:
inicio_execucao = perf_counter()

mascara = (
    (df["tpep_pickup_datetime"] >= "2026-01-15 08:00:00") &
    (df["tpep_pickup_datetime"] < f"2026-01-15 09:00:00")
)
corridas = df.loc[mascara, ["tpep_pickup_datetime", "tpep_dropoff_datetime"]].sort_values("tpep_pickup_datetime").reset_index(drop=True)

inicios = corridas["tpep_pickup_datetime"].to_numpy()
finais = corridas["tpep_dropoff_datetime"].to_numpy()
total = 0
n = len(corridas)

for i in range(n):
    total += ((inicios[i] < finais[i + 1:]) & (inicios[i + 1:] < finais[i])).sum()

print(f"Pandas: 08h–09h | Corridas: {n} | Pares sobrepostos: {total} | Tempo: {perf_counter() - inicio_execucao:.5f}s")


inicio_execucao = perf_counter()

mascara = (
    (df["tpep_pickup_datetime"] >= "2026-01-15 08:00:00") &
    (df["tpep_pickup_datetime"] < f"2026-01-15 10:00:00")
)
corridas = df.loc[mascara, ["tpep_pickup_datetime", "tpep_dropoff_datetime"]].sort_values("tpep_pickup_datetime").reset_index(drop=True)

inicios = corridas["tpep_pickup_datetime"].to_numpy()
finais = corridas["tpep_dropoff_datetime"].to_numpy()
total = 0
n = len(corridas)

for i in range(n):
    total += ((inicios[i] < finais[i + 1:]) & (inicios[i + 1:] < finais[i])).sum()

print(f"Pandas: 08h–10h | Corridas: {n} | Pares sobrepostos: {total} | Tempo: {perf_counter() - inicio_execucao:.5f}s")


inicio_execucao = perf_counter()

mascara = (
    (df["tpep_pickup_datetime"] >= "2026-01-15 08:00:00") &
    (df["tpep_pickup_datetime"] < f"2026-01-15 12:00:00")
)
corridas = df.loc[mascara, ["tpep_pickup_datetime", "tpep_dropoff_datetime"]].sort_values("tpep_pickup_datetime").reset_index(drop=True)

inicios = corridas["tpep_pickup_datetime"].to_numpy()
finais = corridas["tpep_dropoff_datetime"].to_numpy()
total = 0
n = len(corridas)

for i in range(n):
    total += ((inicios[i] < finais[i + 1:]) & (inicios[i + 1:] < finais[i])).sum()

print(f"Pandas: 08h–12h | Corridas: {n} | Pares sobrepostos: {total} | Tempo: {perf_counter() - inicio_execucao:.5f}s")
